In [ ]:
%pip install jmetalpy

In [11]:
import numpy as np

def fast_aco(dist_matrix, n_ants=20, n_iterations=100, alpha=1, beta=2, decay=0.5):
    n_cities = dist_matrix.shape[0]
    # Initialize pheromones
    pheromone = np.ones((n_cities, n_cities)) / n_cities

    # Pre-calculate visibility (1/distance) to save time, avoid div by zero
    visibility = 1.0 / (dist_matrix + np.eye(n_cities) * 1e-9)

    best_path = None
    best_dist = float('inf')

    for _ in range(n_iterations):
        paths = []
        distances = []

        for ant in range(n_ants):
            current_node = np.random.randint(n_cities)
            visited = [current_node]
            path_dist = 0

            # Use a mask for visited nodes (much faster than 'if node in visited')
            mask = np.ones(n_cities, dtype=bool)
            mask[current_node] = False

            for _ in range(n_cities - 1):
                # Calculate probabilities for next city using vector math
                probs = (pheromone[current_node] ** alpha) * (visibility[current_node] ** beta)
                probs[~mask] = 0 # Set already visited cities to 0 probability

                probs /= probs.sum()

                # Choose next city
                next_node = np.random.choice(n_cities, p=probs)

                path_dist += dist_matrix[current_node, next_node]
                visited.append(next_node)
                mask[next_node] = False
                current_node = next_node

            # Close the loop
            path_dist += dist_matrix[visited[-1], visited[0]]
            paths.append(visited)
            distances.append(path_dist)

            if path_dist < best_dist:
                best_dist = path_dist
                best_path = visited

        # Vectorized Pheromone Update
        pheromone *= (1 - decay) # Evaporation
        for path, d in zip(paths, distances):
            for i in range(n_cities - 1):
                pheromone[path[i], path[i+1]] += 1.0 / d
            pheromone[path[-1], path[0]] += 1.0 / d

    return best_path, best_dist

# Quick Test with a 10x10 random matrix
size = 10
coords = np.random.rand(size, 2) * 100
dist_mat = np.sqrt(((coords[:, np.newaxis] - coords[np.newaxis, :]) ** 2).sum(axis=2))

route, dist = fast_aco(dist_mat)
print(f"Optimal Distance: {dist:.2f}")
print(f"Route: {route}")

Optimal Distance: 289.41
Route: [3, 9, 4, 8, 0, 2, 5, 1, 6, 7]
